# 08 -- Documentation, validation & monitoring pack

**What this notebook does (plain English):** The write-up a model-validation or
consulting team would hand over: what we built, how, what the results were, the
limitations, and how we'd **monitor** the models once live. It includes a
**stability check (PSI)** -- a standard early-warning gauge that flags when the
loans coming through the door no longer look like the ones a model was built on.

**Headline result:** the population shifts materially between the calm and crisis
books (high PSI), exactly the kind of drift monitoring is designed to catch.

## Model development summary

**Objective.** Quantify mortgage credit risk end-to-end -- PD, LGD, EAD,
Expected Loss and a downturn stress test -- on the Freddie Mac Single-Family
Loan-Level Dataset.

**Data.** 50,000-loan samples for the 2007, 2008 (crisis) and 2015 (calm)
origination years; origination characteristics joined to monthly performance and
collapsed to one row per loan. Raw data is not redistributed in this repo.

**Methodology.**
- *Default* = first month at 180+ days past due, or a credit-event zero-balance
  code (third-party sale, short sale/charge-off, REO disposition, note sale).
- *PD* = logistic regression on origination features (interpretable scorecard).
- *LGD* = two-stage model (P(loss) x severity) on **realised** losses from
  defaulted, disposed loans; reconciled to Freddie Mac's own loss field (corr ~0.99).
  Extended to **economic (discounted) loss**, an **APRA capital view** (MI excluded,
  20% high-LVR reduction, 20% floor), a margin of conservatism, a downturn LGD, and an
  **independent LGD validation** (notebook 04b) -- see the framework-alignment notes below.
- *EAD* = outstanding balance at default (no CCF -- a term loan has no undrawn limit).
- *EL* = PD x LGD x EAD, staged under IFRS 9 / AASB 9.
- *Stress* = downturn multipliers observed in the 2007/2008 crisis vintages.

**Results.** Default rate ~14% / 7% / 2% and LGD ~58% / 54% / 25% across
2007 / 2008 / 2015; PD model AUC ~0.75-0.80; Expected Loss concentrated in the
crisis vintages and Stage 3.

**Limitations.** Portfolio demonstration, not a regulatory-capital model; US
agency mortgages, not an APRA IRB portfolio; 50k-loan samples, illustrative
calibration; macro stress is scenario-based, not a fitted macroeconomic model.

**Governance / monitoring.** Track discrimination (AUC/Gini/KS), calibration, and
**population stability (PSI)** over time; re-fit on a trigger; maintain model
documentation and an owner for each model.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and re-fit the PD model to get a score to monitor.
import pandas as pd
from src import models, metrics
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
pd_model, pd_cols = models.fit_pd(base)
base = base.copy()
base['pd_hat'] = models.predict_pd(pd_model, pd_cols, base)

In [3]:
# PSI: compare the calm 2015 book (expected) to the crisis books (actual) on
# both a raw driver (credit score) and the model output (PD).
calm = base[base['vintage_year'] == 2015]
crisis = base[base['vintage_year'].isin([2007, 2008])]
psi_tbl = pd.DataFrame([
    {'feature': 'credit_score', 'psi_2015_vs_crisis': round(metrics.psi(calm['credit_score'], crisis['credit_score']), 4)},
    {'feature': 'pd_hat', 'psi_2015_vs_crisis': round(metrics.psi(calm['pd_hat'], crisis['pd_hat']), 4)},
])
psi_tbl['interpretation'] = psi_tbl['psi_2015_vs_crisis'].apply(
    lambda v: 'stable (<0.10)' if v < 0.10 else ('watch (0.10-0.25)' if v < 0.25 else 'material shift (>0.25)'))
save_csv(psi_tbl, 'output/08_monitoring_psi.csv')
psi_tbl

,feature,psi_2015_vs_crisis,interpretation
0,credit_score,0.0995,stable (<0.10)
1,pd_hat,1.5507,material shift (>0.25)


**Reading the table:** PSI above 0.25 signals the incoming population has
shifted materially from the reference book -- here, the crisis vintages look
very different from the calm one, which would trigger a model review in
production. This is the kind of monitoring that keeps a deployed model honest.

## Framework alignment notes (APS 113 / APG 113 / Basel / WP14)

These notes record where the build sits against the regulatory framework. The LGD
work (notebooks 01, 04, 04b, 06) now implements economic-loss discounting, the
APRA-view overlays (MI exclusion, 20% reduction, 20% floor), a margin of
conservatism, downturn LGD, a Stage 3 best estimate, and an independent LGD
validation. The remaining items below are **documentation choices**, stated
explicitly so a reviewer can see they were considered.

**Default definition (Step 2 / APS 113 Att D para 5).** This project defines default
at **180 days past due** (status 6+) or a credit-event zero-balance code. 180 DPD is a
common *mortgage* convention and is what the Freddie Mac data cleanly supports. The
APS 220 / Basel reference point is **90 DPD**; we treat the 180-DPD choice as a "broad
equivalence" adjustment and note that a 90-DPD definition would classify more loans as
defaulted earlier (higher PD, generally lower average LGD as more cures are captured).
A 90-DPD sensitivity flag could be added from the same delinquency field if required.

**Cure rules (Step 2).** A loan is counted as defaulted if it *ever* reached 180+ DPD
within its observed history; we do **not** net out subsequent cures in the default flag
(an observed-to-date definition). The resolution-bias analysis in notebook 04 (P2-1)
shows the practical effect: a large share of 180-DPD "defaults" cure or prepay with
little loss, which is why the cure-aware best estimate sits well below the disposed-only
LGD. A production model would add an explicit cure/re-default (probation) window.

**Borrower/collateral correlation (Part 4.1).** In mortgages, falling house prices both
*trigger* defaults (negative equity) and *deepen* losses (smaller recovery on sale), so
PD and LGD are positively correlated in a downturn. We do not model that correlation
parametrically; instead the **downturn LGD** (notebook 04/06) and the joint PD-and-LGD
stress (notebook 07) are the conservative treatment of it -- both drivers worsen together.

**Observation period (Step 7).** Three discrete vintages (2007, 2008, 2015) is **short
of a full economic cycle**, though it deliberately spans both a severe downturn and a
calm year. This short window is precisely why the **margin of conservatism** (P2-2) is
applied and why benchmarking/qualitative review carry extra weight (APG 113 para 140(c)).

**Segmentation (Step 1).** Current severity drivers are LTV, credit score, loan size
(UPB) and the downturn indicator. In production the LGD segmentation would extend to
**loan purpose, occupancy, and with/without-LMI** (and likely geography/house-price
region), each of which plausibly shifts recovery; they are noted here as the natural
next segments rather than fitted, given the sample size.